# Model 1 feature engineering (JDSearch)

Builds one row per (slate, candidate) with the Model 1 features and saves them to `features/`.

**Inputs** (in `data/`, not committed to git): `train_sampled.parquet`, `val.parquet`, `test.parquet`, `product_metadata_clean.parquet` from the preprocessing notebook.

**Outputs:** `features/features_train.parquet`, `features_val.parquet`, `features_test.parquet`.

**Run order:** Restart kernel, then run all cells top to bottom.

**Design notes**
- Train uses the negative-sampled slates; val and test keep full slates.
- Popularity is built from **train history only**, so val/test information never leaks into features.
- **History is used as given: no recency filter** (`FILTER_MODE = "none"`). We tested dropping recent events (all of them, or only those on the search's own candidates). The filtered variants and their effect are reported in section 10 as a sensitivity check. The filter code stays in the notebook so the check can be rerun.
- Each action type is only used inside its observation window (CLICK 7 days, CART 90 days, ORD 365 days).

## 0. Imports and config

In [1]:
import os, math
from collections import Counter, defaultdict
import numpy as np, pandas as pd, pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

SEP = "\x18"
WEIGHT = {"CLICK": 1, "FLW": 2, "CART": 5, "ORD": 10}
W_LIST = [1, 3, 7]               # recent weighted-affinity windows in days (14 dropped: clicks only exist for ~7 days)
TYPE_WIN = {"CLICK": 7, "CART": 90, "ORD": 365}   # each action type only within its valid observation window
META_COLS = ["wid", "name", "brand_id", "cate_id_3", "cate_id_4"]
FILTER_MODE = "none"       # final choice: use history as given.
                           # "in_slate": drop recent events only if the item is one of the slate's candidates
                           # "all": drop every recent event | "none": keep everything
MIN_AGE_DAYS = 0.05        # only used when FILTER_MODE != "none" ("recent" = closer to the query than this many days)
PROFILE_PER_BATCH = 3            # products sampled per category per batch
PROFILE_CAP = 200                # max products per category in the profile

DATA = "data"       # folder with the cleaned parquet files (git-ignored)
OUT = "features"    # output folder for the feature tables (git-ignored)
os.makedirs(OUT, exist_ok=True)

## 1. Loader

In [2]:
def load(path, columns=None):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)

## 2. Explode list columns into one row per candidate / history action

In [3]:
def explode_cand(df):
    lens = df["candidate_wid_list"].map(len).to_numpy()
    out = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "wid": np.concatenate(df["candidate_wid_list"].to_numpy()).astype(np.int64),
        "label": np.concatenate(df["candidate_label_list"].to_numpy()).astype(np.int8),
        "cand_has_meta": np.concatenate(df["candidate_has_metadata"].to_numpy()).astype(np.int8),
    })
    assert len(out) == lens.sum()
    return out


def explode_hist(df):
    lens = df["history_wid_list"].map(len).to_numpy()
    typ = np.concatenate(df["history_type_list"].to_numpy()).astype(str)
    days = np.concatenate(df["history_days_before"].to_numpy()).astype(np.float32)
    assert len(typ) == len(days) == lens.sum(), "history list lengths disagree"
    out = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "wid": np.concatenate(df["history_wid_list"].to_numpy()).astype(np.int64),
        "typ": typ, "days": days})
    out["w"] = out["typ"].map(WEIGHT).fillna(1).astype(np.int16)
    return out

In [4]:
def slate_keys(df):
    """One int64 key per (slate, candidate): slate_id * 2**32 + wid.
    For train pass the FULL candidate lists (train.parquet), not the negative-sampled ones."""
    lens = df["candidate_wid_list"].map(len).to_numpy()
    sid = np.repeat(df["slate_id"].to_numpy().astype(np.int64), lens)
    wid = np.concatenate(df["candidate_wid_list"].to_numpy()).astype(np.int64)
    return sid * (1 << 32) + wid


def clean_history(h, keys, mode=None, min_age=None):
    """h: exploded history. keys: slate_keys of the slate's FULL candidate list.
    'all'      : drop every event younger than min_age days
    'in_slate' : drop events younger than min_age days only if the item is one of the slate's candidates
    'none'     : keep everything"""
    mode = FILTER_MODE if mode is None else mode
    min_age = MIN_AGE_DAYS if min_age is None else min_age
    if mode == "none":
        return h
    recent = h["days"].to_numpy() < min_age
    if mode == "all":
        return h[~recent]
    if mode == "in_slate":
        hk = h["slate_id"].to_numpy().astype(np.int64) * (1 << 32) + h["wid"].to_numpy()
        return h[~(recent & np.isin(hk, keys))]
    raise ValueError("mode must be 'in_slate', 'all' or 'none'")

## 3. Metadata scan and category profile
One streaming pass over the catalogue, plus the text statistics used by the cosine similarities.

In [5]:
def scan_metadata(path, cand_wids, hist_wids):
    """One streaming pass over the catalogue.
    Returns (meta, profile_rows):
      meta: wid -> brand_id, cate_id_4, (name only for candidate wids)
      profile_rows: sample of (cate_id_4, name) used for the category profile."""
    cand_wids = np.unique(cand_wids)
    need = np.unique(np.concatenate([cand_wids, hist_wids]))
    keep, prof = [], []
    pf = pq.ParquetFile(path)
    for b in pf.iter_batches(batch_size=1_000_000, columns=META_COLS):
        d = b.to_pandas(ignore_metadata=True)
        # category-profile sample (before filtering): a few products per cate_id_4
        ok = d[d["cate_id_4"] >= 0]
        prof.append(ok.groupby("cate_id_4", sort=False).head(PROFILE_PER_BATCH)[["cate_id_4", "name"]])
        m = np.isin(d["wid"].to_numpy(), need)
        d = d[m]
        is_cand = np.isin(d["wid"].to_numpy(), cand_wids)
        d.loc[~is_cand, "name"] = None
        keep.append(d[["wid", "name", "brand_id", "cate_id_4"]])
    meta = pd.concat(keep).drop_duplicates("wid").set_index("wid")
    for c in ["brand_id", "cate_id_4"]:
        meta[c] = meta[c].where(meta[c] >= 0)          # -1 -> missing
    prof = pd.concat(prof)
    prof = prof.groupby("cate_id_4", sort=False).head(PROFILE_CAP)
    return meta, prof


def build_cate_profile(prof):
    """Text statistics for the cosine similarities, from the sampled catalogue names.
    returns dict:
      cate: cate_id_4 -> {term: tf*idf weight} (top 300 terms)      -> category vector
      norm: cate_id_4 -> L2 norm of that vector
      idf : term -> product-level idf (for the name cosine); idf_default for unseen terms"""
    tf = defaultdict(Counter); n = Counter(); df_prod = Counter(); n_prod = 0
    for cate, name in zip(prof["cate_id_4"].to_numpy(), prof["name"]):
        if name is None or len(name) == 0:
            continue
        terms = set(map(str, name))
        n[cate] += 1; n_prod += 1; df_prod.update(terms)
        for t in terms:
            tf[cate][t] += 1
    df_cate = Counter()
    for cate in tf:
        df_cate.update(tf[cate].keys())
    N = max(len(tf), 1)
    cate_w, norm = {}, {}
    for cate, cnt in tf.items():
        w = {t: (c / n[cate]) * math.log((1 + N) / (1 + df_cate[t])) for t, c in cnt.items()}
        top = dict(sorted(w.items(), key=lambda kv: -kv[1])[:300])
        nrm = math.sqrt(sum(v * v for v in top.values()))
        if nrm > 0:
            cate_w[cate], norm[cate] = top, nrm
    idf = {t: math.log((1 + n_prod) / (1 + d)) + 1 for t, d in df_prod.items()}
    return {"cate": cate_w, "norm": norm, "idf": idf, "idf_default": math.log(1 + n_prod) + 1}

## 4. Feature functions

In [ ]:
def query_terms(q):
    """Cleaned data stores the query as an array of term-id strings; raw data as one \x18-joined string."""
    if isinstance(q, str):
        return [t for t in q.split(SEP) if t != ""]
    return [str(t) for t in q if str(t) != ""]


def add_base(c, df):
    qterms = df["query"].map(query_terms)
    q = pd.DataFrame({"slate_id": df["slate_id"].to_numpy(),
                      "current_query_length": qterms.map(len).to_numpy(), "qterms": qterms.to_numpy()})
    return c.merge(q, on="slate_id", how="left")


def build_popularity(tr, keys=None, mode=None, min_age=None):
    """TRAIN-history-only popularity: all actions, ORD only (full year), CART only (90 days).
    keys = slate_keys of the FULL train candidate lists (train.parquet)."""
    h = explode_hist(tr)
    if keys is None:
        keys = slate_keys(tr)
    h = clean_history(h, keys, mode, min_age)
    wid, typ = h["wid"].to_numpy(), h["typ"].to_numpy()
    return {"all": pd.Series(wid).value_counts(),
            "ord": pd.Series(wid[typ == "ORD"]).value_counts(),
            "cart": pd.Series(wid[typ == "CART"]).value_counts()}


def add_popularity(c, pop):
    c["candidate_popularity_count"] = c["wid"].map(pop["all"]).fillna(0).astype(np.float32)
    c["candidate_log_popularity"] = np.log1p(c["candidate_popularity_count"])
    c["candidate_log_pop_ord"] = np.log1p(c["wid"].map(pop["ord"]).fillna(0)).astype(np.float32)
    c["candidate_log_pop_cart"] = np.log1p(c["wid"].map(pop["cart"]).fillna(0)).astype(np.float32)
    return c


def add_type_features(c, h):
    """Per-action-type counts inside each type's own window, for the same item / brand / cate_4,
    plus days since the last CART / ORD of the same item / brand (NaN = never)."""
    for key, short in [("wid", "item"), ("brand_id", "brand"), ("cate_id_4", "cate")]:
        hk = h[h[key].notna()]
        missing = c[key].isna().to_numpy()
        base = pd.DataFrame({"slate_id": c["slate_id"].to_numpy(),
                             key: c[key].fillna(-1).astype(np.int64).to_numpy()})
        for typ, win in TYPE_WIN.items():
            s = hk[(hk["typ"] == typ) & (hk["days"] <= win)]
            g = s.groupby(["slate_id", key])["days"].agg(["size", "min"]).reset_index()
            g[key] = g[key].astype(np.int64)
            m = base.merge(g, on=["slate_id", key], how="left")
            cnt = np.nan_to_num(m["size"].to_numpy(dtype=float), nan=0.0)
            cnt[missing] = np.nan
            c[f"{short}_{typ.lower()}_cnt_{win}d"] = cnt.astype(np.float32)
            if typ in ("CART", "ORD") and short in ("item", "brand"):
                d = m["min"].to_numpy(dtype=float); d[missing] = np.nan
                c[f"{short}_days_since_{typ.lower()}"] = d.astype(np.float32)
    return c


def add_recent_affinity(c, h):
    for key, short in [("brand_id", "brand"), ("cate_id_4", "cate")]:
        hk = h[h[key].notna()]
        ck = c[key].fillna(-1).astype(np.int64)
        for W in W_LIST:
            s = hk[hk["days"] <= W].groupby(["slate_id", key])["w"].sum().rename("v").reset_index()
            s[key] = s[key].astype(np.int64)
            tmp = pd.DataFrame({"slate_id": c["slate_id"].to_numpy(), key: ck.to_numpy()})
            v = tmp.merge(s, on=["slate_id", key], how="left")["v"].to_numpy()
            v = np.where(np.isnan(v), 0.0, v)
            v = np.where(c[key].isna().to_numpy(), np.nan, v)      # NaN only if candidate lacks metadata
            c[f"recent_{short}_affinity_{W}d"] = v.astype(np.float32)
    return c


def add_similarity(c, prof_w):
    """Cosine similarities between the query and (a) the candidate's name, (b) its cate_id_4 profile.
    Query is a binary term vector; name uses idf-weighted terms; category uses the tf*idf profile."""
    idf, d0 = prof_w["idf"], prof_w["idf_default"]
    wt2 = lambda t: idf.get(t, d0) ** 2
    qsets = [frozenset(map(str, q)) for q in c["qterms"].to_numpy()]
    nsets = [frozenset(map(str, n)) if n is not None and len(n) else frozenset() for n in c["name"]]
    qnorm_cache = {}
    name_sim = np.full(len(c), np.nan, dtype=np.float32)
    for i, (qs, ns) in enumerate(zip(qsets, nsets)):
        if not qs or not ns:
            continue
        if qs not in qnorm_cache:
            qnorm_cache[qs] = math.sqrt(sum(wt2(t) for t in qs))
        nn = math.sqrt(sum(wt2(t) for t in ns))
        if qnorm_cache[qs] > 0 and nn > 0:
            name_sim[i] = sum(wt2(t) for t in (qs & ns)) / (qnorm_cache[qs] * nn)
    c["query_candidate_name_sim"] = name_sim
    cache, out = {}, np.full(len(c), np.nan, dtype=np.float32)
    cw, cn = prof_w["cate"], prof_w["norm"]
    for i, (qs, cate) in enumerate(zip(qsets, c["cate_id_4"].to_numpy())):
        if not qs or np.isnan(cate) or cate not in cw:
            continue
        k = (qs, cate)
        if k not in cache:
            p = cw[cate]
            cache[k] = sum(p.get(t, 0.0) for t in qs) / (math.sqrt(len(qs)) * cn[cate])
        out[i] = cache[k]
    c["query_candidate_cate4_sim"] = out
    return c


def build_features(df, meta, pop, prof_w, keys=None, mode=None, min_age=None):
    c = explode_cand(df)
    c = c.join(meta[["name", "brand_id", "cate_id_4"]], on="wid")
    # trust the teammate's flag as the definition of 'has metadata'
    for col in ["name", "brand_id", "cate_id_4"]:
        if col == "name":
            c.loc[c["cand_has_meta"] == 0, col] = None
        else:
            c.loc[c["cand_has_meta"] == 0, col] = np.nan
    h = explode_hist(df)
    h = clean_history(h, slate_keys(df) if keys is None else keys, mode, min_age)   # replaces: h = h[h["days"] >= MIN_AGE_DAYS]
    h = h.join(meta[["brand_id", "cate_id_4"]], on="wid")
    c = add_base(c, df)
    c = add_popularity(c, pop)
    c = add_recent_affinity(c, h)
    c = add_type_features(c, h)
    c = add_similarity(c, prof_w)
    feats = (["current_query_length", "cand_has_meta", "candidate_popularity_count", "candidate_log_popularity"]
             + [f"recent_{s}_affinity_{W}d" for s in ("brand", "cate") for W in W_LIST]
             + ["candidate_log_pop_ord", "candidate_log_pop_cart"]
             + [f"{k}_{t.lower()}_cnt_{w}d" for k in ("item", "brand", "cate") for t, w in TYPE_WIN.items()]
             + [f"{k}_days_since_{t}" for k in ("item", "brand") for t in ("cart", "ord")]
             + ["query_candidate_name_sim", "query_candidate_cate4_sim"])
    out = c[["slate_id", "wid", "label"] + feats].rename(columns={"cand_has_meta": "candidate_has_metadata"})
    return out

## 5. Load the splits and check alignment

In [7]:
splits = {"train": load(f"{DATA}/train_sampled.parquet"),
          "val": load(f"{DATA}/val.parquet"),
          "test": load(f"{DATA}/test.parquet")}
for k, d in splits.items():
    print(k, len(d), "slates")
print("total slates:", sum(len(d) for d in splits.values()))
tr = splits["train"]

# every list column must line up within a row
for k, d in splits.items():
    assert (d["candidate_wid_list"].map(len) == d["candidate_label_list"].map(len)).all()
    assert (d["candidate_wid_list"].map(len) == d["candidate_has_metadata"].map(len)).all()
    assert (d["history_wid_list"].map(len) == d["history_type_list"].map(len)).all()
    assert (d["history_wid_list"].map(len) == d["history_days_before"].map(len)).all()
print("list lengths aligned")

# keys of each split's candidates: only used when FILTER_MODE != "none" (sensitivity check in section 10).
# For train these come from the negative-sampled lists, so the in_slate filter is slightly weaker on train than on val/test.
full_train = load(f"{DATA}/train_sampled.parquet", columns=["slate_id", "candidate_wid_list"])
keys = {"train": slate_keys(full_train), "val": slate_keys(splits["val"]), "test": slate_keys(splits["test"])}
del full_train

train 139882 slates
val 16924 slates
test 17025 slates
total slates: 173831
list lengths aligned


## 6. Train-only popularity

In [8]:
pop = build_popularity(tr, keys["train"])
print({k: len(v) for k, v in pop.items()}, "distinct products by action type (train history)")

{'all': 4768869, 'ord': 1732609, 'cart': 3154679} distinct products by action type (train history)


## 7. Scan the catalogue and build the category profile
Slow (a few minutes) and memory-heavy. Only needs to run once per kernel session.

In [9]:
cand_w = np.concatenate([np.concatenate(d["candidate_wid_list"].to_numpy()) for d in splits.values()])
hist_w = np.unique(np.concatenate([np.concatenate(d["history_wid_list"].to_numpy()) for d in splits.values()]))
meta, prof = scan_metadata(f"{DATA}/product_metadata_clean.parquet", cand_w, hist_w)
prof_w = build_cate_profile(prof)
print(len(meta), "products with metadata matched;", len(prof_w["cate"]), "categories profiled;",
      len(prof_w["idf"]), "terms with idf")

7784366 products with metadata matched; 7137 categories profiled; 186207 terms with idf


## 8. Build and save features

In [10]:
feat = {}
for name, df in splits.items():
    f = build_features(df, meta, pop, prof_w, keys[name])
    f.to_parquet(f"{OUT}/features_{name}.parquet", index=False)
    feat[name] = f
    print(name, f.shape, f"purchase rate {f['label'].eq(3).mean():.4%}")

train (2988641, 30) purchase rate 0.5618%
val (1511179, 30) purchase rate 0.1311%
test (1500706, 30) purchase rate 0.1376%


## 9. Sanity checks

### 9.1 Missing rates
The affinity, brand/category and similarity columns should be missing only for candidates without metadata (about 0.45%). The `days_since_*` columns are mostly empty by design (NaN = never).

In [11]:
for s, f in feat.items():
    print("=====", s, f.shape, "slates:", f.slate_id.nunique())
    print(f.isna().mean().round(4).to_string())

===== train (2988641, 30) slates: 139882
slate_id                      0.0000
wid                           0.0000
label                         0.0000
current_query_length          0.0000
candidate_has_metadata        0.0000
candidate_popularity_count    0.0000
candidate_log_popularity      0.0000
recent_brand_affinity_1d      0.0045
recent_brand_affinity_3d      0.0045
recent_brand_affinity_7d      0.0045
recent_cate_affinity_1d       0.0045
recent_cate_affinity_3d       0.0045
recent_cate_affinity_7d       0.0045
candidate_log_pop_ord         0.0000
candidate_log_pop_cart        0.0000
item_click_cnt_7d             0.0000
item_cart_cnt_90d             0.0000
item_ord_cnt_365d             0.0000
brand_click_cnt_7d            0.0045
brand_cart_cnt_90d            0.0045
brand_ord_cnt_365d            0.0045
cate_click_cnt_7d             0.0045
cate_cart_cnt_90d             0.0045
cate_ord_cnt_365d             0.0045
item_days_since_cart          0.9939
item_days_since_ord           0.99

### 9.2 Consistency checks

In [12]:
# nested windows must be monotone (1d <= 3d <= 7d)
for s, f in feat.items():
    for short in ("brand", "cate"):
        cols = [f"recent_{short}_affinity_{W}d" for W in W_LIST]
        v = f[cols].fillna(0).to_numpy()
        assert (np.diff(v, axis=1) >= 0).all(), (s, short)
print("nested windows OK")

# missing metadata must give missing similarity / affinity
for s, f in feat.items():
    nm = f["candidate_has_metadata"].eq(0)
    assert f.loc[nm, "query_candidate_name_sim"].isna().all()
    assert f.loc[nm, "recent_brand_affinity_3d"].isna().all()
    print(s, "no-metadata share:", round(nm.mean() * 100, 3), "%")

nested windows OK
train no-metadata share: 0.454 %
val no-metadata share: 0.449 %
test no-metadata share: 0.459 %


### 9.3 Similarity features: distribution and purchase gap

In [13]:
f = feat["val"]
for c in ["query_candidate_name_sim", "query_candidate_cate4_sim"]:
    print(c, f[c].describe().round(4).to_dict())
    print("  share exactly 0:", (f[c] == 0).mean().round(3))
print(f.groupby(f.label.eq(3))[["query_candidate_name_sim", "query_candidate_cate4_sim"]].mean())

query_candidate_name_sim {'count': 1504397.0, 'mean': 0.2427, 'std': 0.1386, 'min': 0.0, '25%': 0.1555, '50%': 0.2506, '75%': 0.3319, 'max': 1.0}
  share exactly 0: 0.105
query_candidate_cate4_sim {'count': 1504397.0, 'mean': 0.2221, 'std': 0.1806, 'min': 0.0, '25%': 0.0493, '50%': 0.2006, '75%': 0.3682, 'max': 0.83}
  share exactly 0: 0.113
       query_candidate_name_sim  query_candidate_cate4_sim
label                                                     
False                  0.242699                   0.222056
True                   0.267706                   0.251694


### 9.4 Single-feature AUC on val
Pooled over all slates, so ranking features can look weaker than they are. Popularity should be the strongest; the similarity and brand/category order-history features should beat 0.5.

In [14]:
f = feat["val"]; y = f["label"].eq(3).astype(int)
for c in [c for c in f.columns if c not in ("slate_id", "wid", "label")]:
    x = f[c].fillna(f[c].median())
    print(f"{c:35s} AUC = {roc_auc_score(y, x):.3f}")

current_query_length                AUC = 0.468
candidate_has_metadata              AUC = 0.501
candidate_popularity_count          AUC = 0.810
candidate_log_popularity            AUC = 0.810
recent_brand_affinity_1d            AUC = 0.534
recent_brand_affinity_3d            AUC = 0.532
recent_brand_affinity_7d            AUC = 0.533
recent_cate_affinity_1d             AUC = 0.510
recent_cate_affinity_3d             AUC = 0.504
recent_cate_affinity_7d             AUC = 0.499
candidate_log_pop_ord               AUC = 0.802
candidate_log_pop_cart              AUC = 0.798
item_click_cnt_7d                   AUC = 0.538
item_cart_cnt_90d                   AUC = 0.532
item_ord_cnt_365d                   AUC = 0.529
brand_click_cnt_7d                  AUC = 0.527
brand_cart_cnt_90d                  AUC = 0.536
brand_ord_cnt_365d                  AUC = 0.563
cate_click_cnt_7d                   AUC = 0.492
cate_cart_cnt_90d                   AUC = 0.515
cate_ord_cnt_365d                   AUC 

### 9.5 No position leak
Candidates are shuffled inside each slate, so the first position should not be special.

In [15]:
first = feat["val"].groupby("slate_id").head(1)
print("purchase rate overall:", feat["val"]["label"].eq(3).mean().round(5),
      "| first position:", first["label"].eq(3).mean().round(5))

purchase rate overall: 0.00131 | first position: 0.00183


### 9.6 Per-slate evaluation
Pooled AUC (9.4) mixes candidates from different slates, but Model 1 only has to rank candidates *within* a slate.
Here each feature ranks every val slate on its own, and we report NDCG@10, HitRate@10, HitRate@1 and MRR,
averaged over slates with at least one purchase.

- `random` = constant score, i.e. random order. A feature is only useful if it beats this row.
- Ties are broken at random (averaged over 5 orders); missing values rank last.
- `*_days_since_*` features rank ascending (more recent = better); all others descending, as hypothesised.
  A feature below `random` therefore points the opposite way to what we assumed.
- Features that are constant within a slate (e.g. `current_query_length`) score exactly like `random`:
  they cannot rank on their own, which pooled AUC hides.

In [16]:
# Per-slate ranking metrics.
# Input: one row per (slate, candidate) with slate_id, label (0 none, 1 click, 2 cart, 3 purchase) and a score column.
# relevance="purchase": gain 1 for label 3, slates with >= 1 purchase.
# relevance="graded":   gain 2**label - 1 (click 1, cart 3, purchase 7), slates with >= 1 positive.
# NaN scores rank last; ties are broken at random and averaged over n_tiebreaks orders.

METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    """Start index of each slate block in an array sorted by slate, and the 1-based rank inside it."""
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True,
                      n_tiebreaks=5, seed=0):
    """One row per scored slate with ndcg@k, hit@k, hit@1 and mrr (averaged over random tie-breaks)."""
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)

    # keep only slates that contain at least one relevant item (others have undefined NDCG)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")

    # sort key: larger key = better; NaN -> -inf so it always goes last
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)

    # ideal ordering (does not depend on ties), for the NDCG denominator
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)

    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))   # primary: slate, then score desc, then random
        g = gain[order]                           # same slate blocks as ideal_order, so start/rank reuse
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first

    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def summarize(per_slate, n_boot=1000, seed=0):
    """Mean of each metric over slates with a 95% bootstrap confidence interval (resampling slates)."""
    rng = np.random.default_rng(seed)
    cols = [c for c in per_slate.columns if c != "slate_id"]
    vals = per_slate[cols].to_numpy()
    idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
    boot = vals[idx].mean(axis=1)                       # (n_boot, n_metrics)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"mean": vals.mean(axis=0), "ci_low": lo, "ci_high": hi}, index=cols)


def evaluate_features(df, cols, lower_is_better=(), k=10, relevance="purchase", n_tiebreaks=5, seed=0):
    """Rank each slate by each column on its own. Adds a 'random' row (constant score) as the floor."""
    tmp = df[["slate_id", "label"]].copy()
    tmp["__random__"] = 0.0
    rows = {}
    for c in ["__random__"] + list(cols):
        tmp["__score__"] = df[c].to_numpy() if c != "__random__" else tmp["__random__"].to_numpy()
        ps = per_slate_metrics(tmp, "__score__", k=k, relevance=relevance,
                               higher_is_better=c not in set(lower_is_better),
                               n_tiebreaks=n_tiebreaks, seed=seed)
        rows["random" if c == "__random__" else c] = ps.drop(columns="slate_id").mean()
    return pd.DataFrame(rows).T


def self_test():
    """Hand-checkable cases. Raises AssertionError if the metric code is wrong."""
    # slate 1: purchase ranked 2nd of 3 -> ndcg = 1/log2(3), hit@1 = 0, mrr = 0.5
    # slate 2: purchase ranked 1st      -> all metrics 1
    # slate 3: no purchase              -> excluded
    d = pd.DataFrame({"slate_id": [1, 1, 1, 2, 2, 3, 3],
                      "label":    [0, 3, 0, 3, 1, 0, 1],
                      "s":        [0.9, 0.5, 0.1, 0.8, 0.2, 0.5, 0.4]})
    ps = per_slate_metrics(d, "s", k=10).set_index("slate_id")
    assert list(ps.index) == [1, 2]
    assert np.isclose(ps.loc[1, "ndcg@10"], 1 / np.log2(3))
    assert ps.loc[1, "hit@1"] == 0 and ps.loc[1, "mrr"] == 0.5 and ps.loc[1, "hit@10"] == 1
    assert (ps.loc[2] == 1).all()
    # direction flip: lower is better puts the 0.5 purchase between 0.1 and 0.9 -> still rank 2
    assert np.isclose(per_slate_metrics(d, "s", higher_is_better=False).set_index("slate_id").loc[1, "mrr"], 0.5)
    # NaN goes last even if the purchase has it
    d2 = d.assign(s=[0.9, np.nan, 0.1, 0.8, 0.2, 0.5, 0.4])
    assert per_slate_metrics(d2, "s").set_index("slate_id").loc[1, "mrr"] == 1 / 3
    # constant score on a 3-item slate with one purchase: expected mrr = (1 + 1/2 + 1/3) / 3
    d3 = pd.DataFrame({"slate_id": [1, 1, 1], "label": [3, 0, 0], "s": [0.0, 0.0, 0.0]})
    m = per_slate_metrics(d3, "s", n_tiebreaks=4000)["mrr"].iloc[0]
    assert abs(m - (1 + 1 / 2 + 1 / 3) / 3) < 0.02
    return True

In [17]:
assert self_test()     # hand-checked toy slates
print("metric self-test OK")

# (slate_id, wid) must be unique so later feature files can be joined safely
for s, f in feat.items():
    n_dup = int(f.duplicated(["slate_id", "wid"]).sum())
    print(s, "duplicate (slate_id, wid) rows:", n_dup)

metric self-test OK
train duplicate (slate_id, wid) rows: 0
val duplicate (slate_id, wid) rows: 0
test duplicate (slate_id, wid) rows: 0


In [18]:
f = feat["val"]
y = f["label"].eq(3).astype(int)
cols = [c for c in f.columns if c not in ("slate_id", "wid", "label")]
lower = [c for c in cols if "days_since" in c]

table = evaluate_features(f, cols, lower_is_better=lower, relevance="purchase")

# pooled AUC from 9.4, for side-by-side comparison (sign flipped for lower-is-better features)
pooled = {c: roc_auc_score(y, (-1 if c in lower else 1) * f[c].fillna(f[c].median())) for c in cols}
table["pooled_auc"] = pd.Series(pooled)

n_slates = f.loc[f["label"].eq(3), "slate_id"].nunique()
print("val slates with a purchase:", n_slates)
print(table.sort_values("ndcg@10", ascending=False).round(4).to_string())

val slates with a purchase: 1930
                            ndcg@10  hit@10   hit@1     mrr  pooled_auc
candidate_popularity_count   0.3462  0.5875  0.1623  0.2919      0.8103
candidate_log_popularity     0.3462  0.5875  0.1623  0.2919      0.8103
candidate_log_pop_cart       0.3403  0.5664  0.1665  0.2910      0.7978
candidate_log_pop_ord        0.3205  0.5502  0.1466  0.2698      0.8016
item_click_cnt_7d            0.1334  0.2077  0.0804  0.1342      0.5382
item_days_since_cart         0.1203  0.1940  0.0693  0.1216      0.5121
item_cart_cnt_90d            0.1194  0.1940  0.0666  0.1203      0.5317
item_days_since_ord          0.1125  0.1852  0.0629  0.1141      0.5018
item_ord_cnt_365d            0.1121  0.1856  0.0615  0.1135      0.5291
recent_brand_affinity_7d     0.1092  0.2114  0.0375  0.1025      0.5332
recent_brand_affinity_3d     0.1088  0.2094  0.0385  0.1024      0.5323
recent_brand_affinity_1d     0.1062  0.2033  0.0381  0.1009      0.5345
brand_click_cnt_7d           0.

In [19]:
# 95% bootstrap CI over slates: is the gap to random larger than noise?
for c in ["candidate_log_popularity", "query_candidate_name_sim", "query_candidate_cate4_sim",
          "recent_brand_affinity_3d", "brand_ord_cnt_365d"]:
    ps = per_slate_metrics(f, c, higher_is_better=c not in lower)
    s = summarize(ps).loc[["ndcg@10", "hit@10"]].round(4)
    print(c, "| ndcg@10", tuple(s.loc["ndcg@10"]), "| hit@10", tuple(s.loc["hit@10"]))

f_r = f.assign(_r=0.0)
print("random", "| ndcg@10", tuple(summarize(per_slate_metrics(f_r, "_r")).loc["ndcg@10"].round(4)))

candidate_log_popularity | ndcg@10 (0.3462, 0.3305, 0.3623) | hit@10 (0.5875, 0.5671, 0.6094)
query_candidate_name_sim | ndcg@10 (0.0729, 0.064, 0.0816) | hit@10 (0.1537, 0.1378, 0.1702)
query_candidate_cate4_sim | ndcg@10 (0.0709, 0.0653, 0.0767) | hit@10 (0.1522, 0.1416, 0.1625)
recent_brand_affinity_3d | ndcg@10 (0.1088, 0.1001, 0.1167) | hit@10 (0.2094, 0.1955, 0.2224)
brand_ord_cnt_365d | ndcg@10 (0.0965, 0.0882, 0.1041) | hit@10 (0.1955, 0.1819, 0.2083)
random | ndcg@10 (0.0667, 0.0617, 0.0714)


In [20]:
# same table with click=1, cart=3, purchase=7 gains, on slates with any positive
graded = evaluate_features(f, cols, lower_is_better=lower, relevance="graded")
print(graded.sort_values("ndcg@10", ascending=False).round(4).to_string())

                            ndcg@10  hit@10   hit@1     mrr
candidate_popularity_count   0.3404  0.6440  0.1957  0.3375
candidate_log_popularity     0.3404  0.6440  0.1957  0.3375
candidate_log_pop_cart       0.3309  0.6253  0.1948  0.3313
candidate_log_pop_ord        0.2821  0.5531  0.1649  0.2887
item_click_cnt_7d            0.1529  0.3121  0.1087  0.1826
item_days_since_cart         0.1263  0.2781  0.0795  0.1513
item_cart_cnt_90d            0.1258  0.2782  0.0779  0.1505
recent_brand_affinity_7d     0.1151  0.2896  0.0489  0.1312
recent_brand_affinity_3d     0.1140  0.2870  0.0494  0.1307
brand_click_cnt_7d           0.1131  0.2853  0.0480  0.1295
recent_brand_affinity_1d     0.1118  0.2833  0.0486  0.1293
brand_days_since_cart        0.1068  0.2744  0.0443  0.1238
brand_cart_cnt_90d           0.1045  0.2714  0.0417  0.1211
query_candidate_name_sim     0.0988  0.2484  0.0405  0.1148
item_days_since_ord          0.0978  0.2454  0.0477  0.1184
recent_cate_affinity_7d      0.0978  0.2

## 10. Sensitivity check: does the history filter matter?
We use history unfiltered. Here the same single-feature ranking (per-slate NDCG@10 on val, purchase slates) is recomputed with filters, to show how much the item-level features rely on events just before the query. The popularity table is held fixed (built without a filter), so only the history features change.

- `none`: the final setting.
- `in_slate 10s`: drop events under 10 s old on the search's own candidates.
- `in_slate 4320s`: the same with a 0.05-day (72-minute) cutoff.
- `all 10s`: drop every event under 10 s old.

A feature that is far above `random` under `none` but close to it under the longer filter depends on events seconds to minutes before the query. Whether those events are the search's own interactions (a leak) or genuine recent behaviour cannot be decided from this table alone.

In [21]:
CONFIGS = [("none", 0.0), ("all", 10/86400), ("in_slate", 10/86400), ("in_slate", 0.05)]
KEY = ["item_ord_cnt_365d", "item_cart_cnt_90d", "item_click_cnt_7d", "brand_ord_cnt_365d",
       "recent_brand_affinity_3d", "item_days_since_ord", "item_days_since_cart"]
lower_key = [c for c in KEY if "days_since" in c]

h_val = explode_hist(splits["val"])
res = {}
for mode, age in CONFIGS:
    fv = build_features(splits["val"], meta, pop, prof_w, keys["val"], mode, age)
    t = evaluate_features(fv, KEY, lower_is_better=lower_key, relevance="purchase")
    row = t["ndcg@10"].drop("random").to_dict()
    row["random"] = t.loc["random", "ndcg@10"]
    row["events_dropped"] = len(h_val) - len(clean_history(h_val, keys["val"], mode, age))
    res[f"{mode} {age*86400:.0f}s"] = row
print(pd.DataFrame(res).T.round(4).to_string())

                item_ord_cnt_365d  item_cart_cnt_90d  item_click_cnt_7d  brand_ord_cnt_365d  recent_brand_affinity_3d  item_days_since_ord  item_days_since_cart  random  events_dropped
none 0s                    0.1121             0.1194             0.1334              0.0965                    0.1088               0.1125                0.1203  0.0667             0.0
all 10s                    0.1089             0.1144             0.1328              0.0960                    0.1082               0.1093                0.1149  0.0667          1295.0
in_slate 10s               0.1089             0.1144             0.1328              0.0956                    0.1073               0.1093                0.1149  0.0667           426.0
in_slate 4320s             0.1039             0.0926             0.0848              0.0943                    0.0907               0.1040                0.0918  0.0667          9728.0
